# Tokens and Embeddings

A language model cannot read text directly. It computes with numbers, so every text first goes through two steps:

1. **Tokenization** splits the text into pieces, called **tokens**, from a fixed vocabulary, and gives each token a number, its **id**.
2. The **embedding lookup** turns each id into a **vector**, a list of numbers the model can compute with.

A simple regular expression such as `\b\w\w+\b` also splits text into words, but it loses information: `ne'er` becomes `ne` and `er`, and `12.5` disappears. That is acceptable for a small classifier, but a language model has to handle any text, including misspellings, numbers and emojis. In this notebook you compare four trained tokenizers that solve this problem, and then look at the vectors the embedding lookup produces.

## Learning Objectives

By the end of this notebook, you should be able to:

- Compare how four tokenizers split the same text.
- Calculate a corpus's cost in tokens per word.
- Measure static and contextual embeddings with cosine similarity.

## Setup

In [ ]:
# standard library
import re
from pathlib import Path

# data stack
import pandas as pd

# deep learning stack
import torch

# nlp stack
import transformers
from transformers import AutoModel, AutoTokenizer

The model hub prints a lot of output: a progress bar for every file it downloads and every weight it loads, and a report of which weights a checkpoint did and did not supply. None of it matters here, and all of it goes to the error stream, where it looks like a traceback.

One note the setting below does not reach is `You are sending unauthenticated requests to the HF Hub`, printed once on the first download. These models are public and need no account; one would only raise the rate limit.

In [ ]:
# Hide the hub's log output. Raise this to `set_verbosity_warning()` if a
# model ever loads and behaves oddly; the report it hides names any missing weights.
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

## 1. Words, Letters, or Something Between

Two obvious ways to cut text, and what each costs:

| | Vocabulary | Unknown input | Sequence length |
|---|---|---|---|
| **One token per word** | hundreds of thousands, and still growing | every misspelling, name and new word is unknown | short |
| **One token per letter** | about a hundred | nothing is ever unknown | very long |

Neither end works. A word vocabulary is never finished, and a model cannot answer a question containing a word it has no number for. A letter vocabulary never meets anything new, but every sentence becomes hundreds of steps long.

In [ ]:
SENTENCE = "The bioluminescence was unexpected."

print("by word:  ", re.findall(r"\b\w\w+\b", SENTENCE.lower()))
print("by letter:", list(SENTENCE)[:20], "...")
print(
    f"{len(re.findall(r'\b\w\w+\b', SENTENCE))} words against {len(SENTENCE)} characters"
)

**Subword** tokenization is the compromise between the two. Common words stay whole, rare ones are cut into pieces that are themselves common, and the pieces are learned from a corpus rather than chosen by hand. The vocabulary is fixed in advance (usually between 30,000 and 50,000 entries), and anything can be spelled out of it.

## 2. What a Trained Tokenizer Does

A tokenizer is trained with its model and ships with it, so you load it by the model's name. [`AutoTokenizer`](https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoTokenizer) fetches the right one from the model hub; the four used here are a few megabytes together and are cached after the first run.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"{bert_tokenizer.vocab_size:,} tokens in the vocabulary")

In [ ]:
# add_special_tokens=False leaves out the markers BERT puts around every input; section 6 shows them
ids = bert_tokenizer(
    "The bioluminescence was unexpected.", add_special_tokens=False
).input_ids

for token, number in zip(bert_tokenizer.convert_ids_to_tokens(ids), ids):
    print(f"{token:12s} {number}")

**Result.** `the`, `was` and `unexpected` are whole words with one number each. `bioluminescence` is four pieces: `bio`, `##lum`, `##ines`, `##cence`.

The `##` means *this continues the word before*. Without it the pieces could not be told from separate words, and `bio` at the start of a word is a different token from `##bio` inside one.

The tokenizer has never seen `bioluminescence`. It has seen all four pieces, and that is the principle the whole method rests on: 30,522 entries, and almost nothing is ever unknown. Section 3 finds the exception.

## 3. Four Tokenizers, One Text

Every model makes its own choices, and the easiest way to see them is to give four tokenizers the same difficult sentence: a possessive, an archaic contraction, a word none of them will know, a decimal number, an emoji and a German noun.

In [ ]:
TEXT = "Dickinson's ne'er-do-well bioluminescence: 12.5 degrees, \N{SPOUTING WHALE} Walfisch."

MODELS = ["bert-base-uncased", "bert-base-cased", "gpt2", "google/flan-t5-small"]
tokenizers = {name: AutoTokenizer.from_pretrained(name) for name in MODELS}
print(f"{len(re.findall(r'\b\w\w+\b', TEXT)):,} words by a regular expression's count")

In [ ]:
rows = []
for name, tokenizer in tokenizers.items():
    ids = tokenizer(TEXT, add_special_tokens=False).input_ids
    rows.append(
        {
            "tokenizer": name,
            "vocabulary": tokenizer.vocab_size,
            "tokens": len(ids),
            # decode turns the numbers back into a string; does it match what went in?
            "rebuilds the text": tokenizer.decode(ids) == TEXT,
        }
    )
pd.DataFrame(rows).set_index("tokenizer")

In [ ]:
for name, tokenizer in tokenizers.items():
    pieces = tokenizer.convert_ids_to_tokens(
        tokenizer(TEXT, add_special_tokens=False).input_ids
    )
    print(f"{name}\n    {' | '.join(pieces)}\n")

**Result.** Nine words become 25 to 30 tokens, and no two tokenizers agree.

- **Case.** `bert-base-uncased` lowercases everything, so `Dickinson` and `dickinson` end as one token. The cased model keeps the difference and pays for it, since its vocabulary has to hold both spellings of every word.
- **Spaces.** GPT-2's `Ġ` and T5's `▁` are the space in front of a token, carried along with it. BERT drops spaces altogether.
- **The emoji.** BERT and T5 have no entry for it and say so: `[UNK]` and `<unk>`. The whale emoji is lost and cannot be recovered. GPT-2 spells it out in bytes, `ĠðŁ`, `Ĳ`, `³`: unreadable, and exactly reversible.
- **Numbers.** `12.5` is `12`, `.`, `5` to BERT and GPT-2, but `1` and `2.5` to T5, a split that makes arithmetic harder for the model that has to do it.

Only GPT-2 rebuilds the text. Its vocabulary is **byte-level**: it has an entry for every one of the 256 possible bytes, so no input is ever unknown and `decode` returns exactly what went in. Dropped spaces and `[UNK]` both cost the others that guarantee, and it is the reason models that have to *write* text are built this way.

## 4. What It Costs

Subwords are not free. A text is always at least as many tokens as it has words, usually more, and a model's price and its context limit are both counted in tokens. The cell below measures the ratio on something real: 200 poems by Emily Dickinson and Walt Whitman, one poem per row.

In [ ]:
poems = pd.read_csv(Path("data") / "poems.csv")
sample = poems["text"].head(200)
words = sample.str.findall(r"\b\w\w+\b").str.len().sum()
print(f"{len(sample)} poems, {words:,} words")

In [ ]:
rows = []
for name, tokenizer in tokenizers.items():
    tokens = sum(
        len(tokenizer(text, add_special_tokens=False).input_ids) for text in sample
    )
    rows.append(
        {
            "tokenizer": name,
            "tokens": tokens,
            "tokens per word": round(tokens / words, 2),
        }
    )
pd.DataFrame(rows).set_index("tokenizer")

**Result.** 12,394 words become between 17,410 and 20,815 tokens: 1.40 tokens per word in BERT's vocabulary, 1.68 in GPT-2's. The cased model costs more than the uncased one, because separating `The` from `the` leaves more words to be spelled out in pieces.

This is the number to remember whenever a model charges by the token or limits its context to a count of them. English runs at roughly one and a half tokens per word, and anything unusual (code, names, another language) runs higher.

**Your turn:** give the tokenizers a sentence of your own and look at where they disagree. Something with a name, a long or technical word, a number or another language in it will separate them fastest.

In [ ]:
# TODO: put your own sentence in `mine`, then print each tokenizer's pieces for it,
# as the cell in section 3 does.
mine = ...
# your code:

## 5. From a Number to a Vector

A token id is a row number and nothing else. `bank` being 2924 says only that it is the 2,924th entry of the vocabulary, and 2925 is not related to it in meaning.

The meaning sits one step further on, in the **embedding matrix**: one row of numbers per vocabulary entry, learned while the model was trained. Looking a token up is the model's first act on any input. The model is about 420 MB, downloaded on the first run.

In [ ]:
bert = AutoModel.from_pretrained("bert-base-uncased")
bert.eval()  # inference mode: no dropout, so the same input always gives the same output

embeddings = bert.get_input_embeddings().weight.detach()
print("embedding matrix:", tuple(embeddings.shape))

In [ ]:
total = sum(p.numel() for p in bert.parameters())
print(f"{embeddings.numel():,} numbers in the lookup table")
print(f"{total:,} parameters in the whole model")
print(f"{embeddings.numel() / total:.0%} of the model is the lookup table")

**Result.** 30,522 rows, one per vocabulary entry, each 768 numbers long: 23.4 million of BERT's 109.5 million parameters, a fifth of the model, before a single layer has run.

768 is this model's **hidden size**: the width every vector inside it has. A larger model is wider as well as deeper, which is most of why it is larger.

In [ ]:
# Take the row for `bank` out of two different sentences and compare them.
river = "I sat on the river bank and watched the water."
money = "She works at the bank downtown and counts money."


def lookup(sentence, word="bank"):
    """Return the embedding row the lookup table holds for `word` in `sentence`."""
    encoded = bert_tokenizer(sentence, return_tensors="pt")
    pieces = bert_tokenizer.convert_ids_to_tokens(encoded.input_ids[0])
    position = pieces.index(word)
    return embeddings[encoded.input_ids[0, position]]

In [ ]:
cosine = torch.nn.functional.cosine_similarity
print(f"{cosine(lookup(river), lookup(money), dim=0):.3f}  the two rows, compared")

**Result.** 1.000, the same vector, to every decimal place. A lookup table has no way to produce anything else: one row per entry, and `bank` is one entry.

So the vector is **static**. The model's whole knowledge of `bank` before it reads anything is one point, averaging a river bank and a savings bank and whatever else the word does, with no way to tell them apart. Pretrained word vectors such as word2vec and GloVe have exactly this shape and exactly this limit.

## 6. The Same Word, Two Sentences

Section 5 showed that the embedding lookup gives `bank` the same vector in every sentence. The lookup is only the model's first step. After it, BERT passes the vectors through **twelve transformer layers**, and each layer updates every token's vector using the other tokens in the sentence. The output is again one vector per token, each 768 numbers long, but the numbers now depend on the sentence around the token. These output vectors are called **contextual** vectors.

The figure shows both steps for two sentences that contain `bank`. Both give `bank` the same id and the same static vector; only after the transformer layers do the two vectors differ:

<img src="assets/text_to_vectors.png" alt="Two sentences containing bank go through the same steps: text, tokens, ids, embedding lookup, twelve transformer layers and a contextual vector. Both get id 2924 and an identical static vector; only after the transformer layers do their vectors differ." width="1000">

The code below runs the full model. Two details differ from the earlier cells:

- `return_tensors="pt"` returns PyTorch tensors instead of Python lists, because the model expects tensors.
- The special tokens left out in section 2 are back, because the model was trained with them: `[CLS]` starts every input and `[SEP]` ends it.

`contextual` returns the output vector of one word in a sentence. The cell after it prints the tokens of the river sentence and the shape of the model's output.

In [ ]:
def contextual(sentence, word="bank"):
    """Return the vector the model produces for `word` after reading `sentence`."""
    encoded = bert_tokenizer(sentence, return_tensors="pt")
    pieces = bert_tokenizer.convert_ids_to_tokens(encoded.input_ids[0])
    position = pieces.index(word)
    with torch.no_grad():  # no gradients needed, and it halves the memory
        output = bert(**encoded)
    return output.last_hidden_state[0, position]

In [ ]:
encoded = bert_tokenizer(river, return_tensors="pt")
print(bert_tokenizer.convert_ids_to_tokens(encoded.input_ids[0]))
with torch.no_grad():
    print("one vector per token:", tuple(bert(**encoded).last_hidden_state.shape))

In [ ]:
# A third sentence, river-sense again, as a control: if the vector tracked the sentence
# rather than the sense, this one would be no closer to `river` than `money` is.
boat = "The boat drifted toward the muddy bank, reeds brushing the hull."

print(
    f"{cosine(contextual(river), contextual(money), dim=0):.3f}  river bank vs. savings bank"
)
print(
    f"{cosine(contextual(river), contextual(boat), dim=0):.3f}  river bank vs. river bank"
)
print(
    f"{cosine(contextual(money), contextual(boat), dim=0):.3f}  savings bank vs. river bank"
)

**Result.** The two senses sit at 0.487. The two river banks sit at 0.810, and the control sentence shares no content word with the first, not `river` and not `water`, so what the vectors matched on was the sense and not the wording.

- The vector that goes **in** is the word: one row, the same every time.
- The vector that comes **out** is the word *in this sentence*: computed fresh from all the other tokens.

Twelve layers did that, and this notebook has not said how. The mechanism is **attention**, and it is the next notebook's subject.

**Your turn:** pick a word of your own that means two things (`bat`, `spring`, `crane`, `light`), write a sentence for each sense, and measure both cosines. Mind the tokenizer: a word that comes out in pieces will not be found by `pieces.index(word)`, so choose one that stays whole, or look for its first piece.

In [ ]:
# TODO: two sentences using the same word in different senses, then print the static
# cosine with `lookup` and the contextual one with `contextual`.
word = ...
sentence_a = ...
sentence_b = ...
# your code:

## Summary

- **Subword** tokenization is the split a language model needs: it can represent **any** input, where a word vocabulary and a hand-written pattern both cannot. Common words stay whole, rare ones become pieces, and 30,000 to 50,000 entries cover everything. `bio ##lum ##ines ##cence` is a word the tokenizer has never seen, spelled out of pieces it has.
- Tokenizers differ in ways that matter: whether case survives, what becomes of characters outside the vocabulary, and whether the original text can be rebuilt exactly. Load the one the model was trained with, by the model's name.
- Tokens, not words, are what context limits and API charges are counted in. English runs at about one and a half tokens per word.
- The **embedding matrix** turns each id into a vector. It is a lookup table, so its vectors are **static**: one per vocabulary entry, carrying every sense of the word at once.
- Running the model turns those into **contextual** vectors: one word, a different vector in every sentence. That is what a transformer is for, and how it does it comes next.

## Sources

- The comparison of trained tokenizers on one difficult text, and the contextual embeddings of section 6, follow *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst (O'Reilly, 2024), chapter 2. Its [companion code](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) is Apache-2.0.
- The tokenizers and `bert-base-uncased` come from the [Hugging Face model hub](https://huggingface.co/); each model's own license applies to it.
- The poems are Project Gutenberg eBooks [#12242](https://www.gutenberg.org/ebooks/12242) and [#1322](https://www.gutenberg.org/ebooks/1322).

The repository [README](README.md) lists every source and license.